# 01 · Ingest

**LCL:** Single wide file `elec_house.csv` — households as columns, timestamps as rows  
**CHP:** Single wide file `chp_data.csv` — two-row header, MultiIndex columns  

**NOTE:** Column names are now explicit per dataset:  
- **LCL:** `consumption_kwh` (residential electricity consumption in kWh)  
- **CHP:** `production_mwh` (electricity generation in MW, reported as hourly MWh equivalent)  

**Output:** `data/output/raw_long_{dataset}.parquet`

In [1]:
import sys, os
_root = os.getcwd()
for _ in range(5):
    if os.path.isdir(os.path.join(_root,'config')) and os.path.isdir(os.path.join(_root,'utils')):
        break
    _root = os.path.dirname(_root)
sys.path.insert(0, _root); os.chdir(_root)

from config.settings import cfg, DATASET
import pandas as pd
import numpy as np

print(f'Dataset : {DATASET.upper()}')

Dataset : LCL


## LCL: Load and reshape

In [2]:
if DATASET == 'lcl':
    raw_file = os.path.join(_root, cfg['RAW_FILE'])
    print(f'Loading: {raw_file}')

    elec_house = pd.read_csv(
        raw_file, index_col=cfg['RAW_TIME_COL'],
        parse_dates=True, low_memory=False
    )
    print(f'Wide shape : {elec_house.shape}')
    print(f'Date range : {elec_house.index.min()} -> {elec_house.index.max()}')

    raw_long = (
        elec_house.reset_index()
        .rename(columns={cfg['RAW_TIME_COL']: 'timestamp'})
        .melt(id_vars='timestamp', var_name=cfg['ID_COL'], value_name=cfg['DEMAND_COL'])
    )
    raw_long['timestamp']       = pd.to_datetime(raw_long['timestamp'])
    raw_long[cfg['DEMAND_COL']] = pd.to_numeric(raw_long[cfg['DEMAND_COL']], errors='coerce')
    raw_long['file']            = 'elec_house'

    print(f'Long shape              : {raw_long.shape}')
    print(f'Unique units            : {raw_long[cfg["ID_COL"]].nunique()}')
    print(f'NaNs in {cfg["DEMAND_COL"]}: {raw_long[cfg["DEMAND_COL"]].isna().sum():,}')
    raw_long.head(3)

Loading: /Users/taliaqaiser/Desktop/PhD/Work/April 2026/hierarchical-forecast/data/raw/lcl/elec_house.csv
Wide shape : (39727, 5561)
Date range : 2011-11-23 09:00:00 -> 2014-02-28 00:00:00


KeyboardInterrupt: 

## CHP: Load and reshape

In [ ]:
if DATASET == 'chp':
    raw_file = os.path.join(_root, cfg['RAW_FILE'])
    raw = pd.read_csv(raw_file, header=None, low_memory=False)
    print(f'Raw shape: {raw.shape}')

    location_ids = pd.Series(raw.iloc[0, 1:].values).ffill().values
    metrics      = raw.iloc[1, 1:].values
    multi_cols   = pd.MultiIndex.from_arrays([location_ids, metrics],
                                              names=['chp_id','metric'])
    data_vals         = raw.iloc[2:, 1:].copy()
    data_vals.columns = multi_cols
    data_vals         = data_vals.astype(float)

    if cfg['DROP_UNIT'] in data_vals.columns.get_level_values('chp_id'):
        data_vals = data_vals.drop(columns=cfg['DROP_UNIT'], level='chp_id')
        print(f'Dropped CHP {cfg["DROP_UNIT"]}')

    data_vals.index = pd.date_range(
        start=cfg['DATETIME_START'], end=cfg['DATETIME_END'], freq=cfg['FREQ']
    )
    data_vals.index.name = 'timestamp'

    production = data_vals.xs(cfg['RAW_DEMAND_COL'],   level='metric', axis=1)
    forecast   = data_vals.xs(cfg['RAW_FORECAST_COL'], level='metric', axis=1)
    capacity   = data_vals.xs(cfg['RAW_CAPACITY_COL'], level='metric', axis=1)

    # CHANGED: Use cfg['DEMAND_COL'] instead of hardcoded 'kWh'
    raw_long = (
        production.reset_index()
        .melt(id_vars='timestamp', var_name='chp_id', value_name=cfg['DEMAND_COL'])
    )
    raw_long['chp_id'] = raw_long['chp_id'].astype(str)

    forecast_long = (
        forecast.reset_index()
        .melt(id_vars='timestamp', var_name='chp_id', value_name='production_mwh_forecast_intraday')
    )
    forecast_long['chp_id'] = forecast_long['chp_id'].astype(str)

    chp_metadata = capacity.median().reset_index()
    chp_metadata.columns = ['chp_id','capacity_mw']
    chp_metadata['chp_id'] = chp_metadata['chp_id'].astype(str)

    raw_long = raw_long.merge(chp_metadata, on='chp_id', how='left')
    print(f'Shape: {raw_long.shape}')
    raw_long.head(3)

## Merge metadata (LCL only)

In [ ]:
if DATASET == 'lcl' and cfg['METADATA_FILE'] is not None:
    meta_path = os.path.join(_root, cfg['METADATA_FILE'])
    metadata  = pd.read_csv(meta_path)[cfg['META_COLS']]
    raw_long  = raw_long.merge(metadata, on=cfg['ID_COL'], how='left')
    print('Metadata merged. Columns:', raw_long.columns.tolist())

Metadata merged. Columns: ['timestamp', 'LCLid', 'kWh', 'file_x', 'Acorn_grouped', 'Acorn', 'stdorToU', 'file_y']


## Summary statistics

In [ ]:
print('Shape          :', raw_long.shape)
print('Unique units   :', raw_long[cfg['ID_COL']].nunique())
print('Date range     :', raw_long['timestamp'].min(), '->', raw_long['timestamp'].max())
print(f'NaNs in demand :', raw_long[cfg['DEMAND_COL']].isna().sum())
dupes = raw_long.duplicated(subset=['timestamp', cfg['ID_COL']]).sum()
print(f'Duplicate pairs:', dupes)
raw_long.head(3)

Shape          : (220921847, 8)
Unique units   : 5561
Date range     : 2011-11-23 09:00:00 -> 2014-02-28 00:00:00
NaNs in demand : 53110386
Duplicate pairs: 0


,timestamp,LCLid,kWh,file_x,Acorn_grouped,Acorn,stdorToU,file_y
0,2011-11-23 09:00:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
1,2011-11-23 09:30:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
2,2011-11-23 10:00:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0


## Save to parquet

In [ ]:
out_dir  = os.path.join(_root, cfg['OUTPUT_DIR'])
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, f'raw_long_{DATASET}.parquet')
raw_long.to_parquet(out_path, index=False)
print(f'Saved -> {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)')

if DATASET == 'chp':
    forecast_long.to_parquet(os.path.join(out_dir,'chp_intraday_forecast.parquet'), index=False)
    chp_metadata.to_parquet(os.path.join(out_dir,'chp_capacity.parquet'), index=False)

check = pd.read_parquet(out_path)
print(f'Read-back: {check.shape} | NaNs: {check[cfg["DEMAND_COL"]].isna().sum():,}')
check.head(10)

Saved -> /Users/taliaqaiser/Desktop/PhD/Work/April 2026/hierarchical-forecast/data/output_lcl/raw_long_lcl.parquet  (755.7 MB)
Read-back: (220921847, 8) | NaNs: 53,110,386


,timestamp,LCLid,kWh,file_x,Acorn_grouped,Acorn,stdorToU,file_y
0,2011-11-23 09:00:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
1,2011-11-23 09:30:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
2,2011-11-23 10:00:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
3,2011-11-23 10:30:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
4,2011-11-23 11:00:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
5,2011-11-23 11:30:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
6,2011-11-23 12:00:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
7,2011-11-23 12:30:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
8,2011-11-23 13:00:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
9,2011-11-23 13:30:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0


In [ ]:
check.head(10)

,timestamp,LCLid,kWh,file_x,Acorn_grouped,Acorn,stdorToU,file_y
0,2011-11-23 09:00:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
1,2011-11-23 09:30:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
2,2011-11-23 10:00:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
3,2011-11-23 10:30:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
4,2011-11-23 11:00:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
5,2011-11-23 11:30:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
6,2011-11-23 12:00:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
7,2011-11-23 12:30:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
8,2011-11-23 13:00:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0
9,2011-11-23 13:30:00,MAC000002,NaN,elec_house,Affluent,ACORN-A,Std,block_0


In [3]:
"""
Fixed LCL ingestion notebook
Problem: Values were all NaN after reading from CSV
Solution: Clean non-numeric characters before converting to float
"""

import pandas as pd
import numpy as np
import sys, os

# Setup
_root = os.getcwd()
for _ in range(5):
    if os.path.isdir(os.path.join(_root,'config')) and os.path.isdir(os.path.join(_root,'utils')):
        break
    _root = os.path.dirname(_root)
sys.path.insert(0, _root); os.chdir(_root)

from config.settings import cfg, DATASET
print(f'Dataset : {DATASET.upper()}')

if DATASET == 'lcl':
    # ─── LCL: Load and reshape with proper cleaning ───
    raw_file = os.path.join(_root, cfg['RAW_FILE'])
    print(f'Loading: {raw_file}')

    # Read as all strings first to inspect
    elec_house = pd.read_csv(
        raw_file, 
        index_col=cfg['RAW_TIME_COL'],
        dtype=str,  # ← KEY: Read everything as string first
        low_memory=False
    )
    print(f'Wide shape : {elec_house.shape}')
    print(f'Date range : {elec_house.index.min()} -> {elec_house.index.max()}')
    
    # Debug: Check what's in the first cell
    sample_value = elec_house.iloc[0, 0]
    print(f'Sample raw value: "{sample_value}" (type: {type(sample_value).__name__})')
    
    # Reshape to long format
    raw_long = (
        elec_house.reset_index()
        .rename(columns={cfg['RAW_TIME_COL']: 'timestamp'})
        .melt(id_vars='timestamp', var_name=cfg['ID_COL'], value_name=cfg['DEMAND_COL'])
    )
    
    # Clean the demand column: remove spaces, commas, quotes before converting
    print(f'\nCleaning {cfg["DEMAND_COL"]} column:')
    print(f'  Before: dtype={raw_long[cfg["DEMAND_COL"]].dtype}, nulls={raw_long[cfg["DEMAND_COL"]].isna().sum()}')
    
    raw_long[cfg['DEMAND_COL']] = (
        raw_long[cfg['DEMAND_COL']]
        .str.strip()  # Remove leading/trailing whitespace
        .str.replace(',', '', regex=False)  # Remove commas (thousands separator)
        .str.replace('"', '', regex=False)  # Remove quotes
        .str.replace("'", '', regex=False)  # Remove apostrophes
        .apply(lambda x: pd.to_numeric(x, errors='coerce'))  # Convert to float
    )
    
    print(f'  After: dtype={raw_long[cfg["DEMAND_COL"]].dtype}')
    print(f'  Non-null: {raw_long[cfg["DEMAND_COL"]].notna().sum():,}')
    print(f'  NaNs: {raw_long[cfg["DEMAND_COL"]].isna().sum():,}')
    print(f'  Sample non-NaN values: {raw_long[raw_long[cfg["DEMAND_COL"]].notna()][cfg["DEMAND_COL"]].head().values}')
    
    raw_long['timestamp'] = pd.to_datetime(raw_long['timestamp'])
    raw_long['file'] = 'elec_house'
    
    print(f'\nLong shape : {raw_long.shape}')
    print(f'Unique units : {raw_long[cfg["ID_COL"]].nunique()}')
    print(f'Non-null {cfg["DEMAND_COL"]}: {raw_long[cfg["DEMAND_COL"]].notna().sum():,}')
    print(f'Date range : {raw_long["timestamp"].min()} -> {raw_long["timestamp"].max()}')
    
    # Merge metadata if available
    if cfg['METADATA_FILE'] is not None:
        meta_path = os.path.join(_root, cfg['METADATA_FILE'])
        metadata = pd.read_csv(meta_path)[cfg['META_COLS']]
        raw_long = raw_long.merge(metadata, on=cfg['ID_COL'], how='left')
        print(f'\nMetadata merged. Columns: {raw_long.columns.tolist()}')
    
    # Save to parquet
    out_dir = os.path.join(_root, cfg['OUTPUT_DIR'])
    os.makedirs(out_dir, exist_ok=True)
    
    out_path = os.path.join(out_dir, f'raw_long_{DATASET}.parquet')
    raw_long.to_parquet(out_path, index=False)
    print(f'\nSaved -> {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)')
    
    # Verify
    check = pd.read_parquet(out_path)
    print(f'Verification read-back:')
    print(f'  Shape: {check.shape}')
    print(f'  Non-null {cfg["DEMAND_COL"]}: {check[cfg["DEMAND_COL"]].notna().sum():,}')
    print(f'  NaNs: {check[cfg["DEMAND_COL"]].isna().sum():,}')
    print(f'  Stats:')
    print(f'    Mean: {check[cfg["DEMAND_COL"]].mean():.4f}')
    print(f'    Std: {check[cfg["DEMAND_COL"]].std():.4f}')
    print(f'    Min: {check[cfg["DEMAND_COL"]].min():.4f}')
    print(f'    Max: {check[cfg["DEMAND_COL"]].max():.4f}')
    print(f'\nFirst 5 rows:')
    print(check.head(5))

Dataset : LCL
Loading: /Users/taliaqaiser/Desktop/PhD/Work/April 2026/hierarchical-forecast/data/raw/lcl/elec_house.csv
Wide shape : (39727, 5561)
Date range : 2011-11-23 09:00:00 -> 2014-02-28 00:00:00
Sample raw value: "nan" (type: float)

Cleaning kWh column:
  Before: dtype=object, nulls=53110386
  After: dtype=float64
  Non-null: 167,811,461
  NaNs: 53,110,386
  Sample non-NaN values: [0. 0. 0. 0. 0.]

Long shape : (220921847, 4)
Unique units : 5561
Non-null kWh: 167,811,461
Date range : 2011-11-23 09:00:00 -> 2014-02-28 00:00:00

Metadata merged. Columns: ['timestamp', 'LCLid', 'kWh', 'file_x', 'Acorn_grouped', 'Acorn', 'stdorToU', 'file_y']

Saved -> /Users/taliaqaiser/Desktop/PhD/Work/April 2026/hierarchical-forecast/data/output_lcl/raw_long_lcl.parquet  (755.7 MB)
Verification read-back:
  Shape: (220921847, 8)
  Non-null kWh: 167,811,461
  NaNs: 53,110,386
  Stats:
    Mean: 0.2118
    Std: 0.2972
    Min: 0.0000
    Max: 10.7610

First 5 rows:
            timestamp      LCLi